# 03 - QSVM Implementation

**QZero**: Quantum Zero-Day Detection System

This notebook implements Quantum Support Vector Machine (QSVM) for zero-day attack detection.

## Contents
1. Quantum Feature Maps
2. QSVM Training
3. Kernel Visualization
4. Comparison with Classical SVM
5. Circuit Analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Qiskit imports
try:
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
    from qiskit.visualization import plot_circuit_layout, circuit_drawer
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    QISKIT_AVAILABLE = True
    print("✅ Qiskit loaded successfully")
except ImportError as e:
    QISKIT_AVAILABLE = False
    print(f"❌ Qiskit not available: {e}")

from src.preprocessing.data_loader import generate_synthetic_data, DataLoader
from src.preprocessing.feature_engineering import FeatureEngineer
from src.quantum.qsvm import QSVM, QSVMConfig, compare_quantum_classical
from src.quantum.feature_maps import (
    create_feature_map, 
    FeatureMapType,
    get_feature_map_info
)

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Data Preparation for Quantum

In [ ]:
# Generate and prepare data
X, y = generate_synthetic_data(n_samples=5000, random_state=42)
loader = DataLoader()
data = loader.prepare_data(X, y, binary=True)

# Reduce to 4 features for quantum (4 qubits)
N_QUBITS = 4
engineer = FeatureEngineer(n_qubits=N_QUBITS, reduction_method='pca')

X_train_q = engineer.fit_transform(data['X_train'], data['y_train'])
X_val_q = engineer.transform(data['X_val'])
X_test_q = engineer.transform(data['X_test'])

print(f"Quantum-ready data shapes:")
print(f"  Train: {X_train_q.shape}")
print(f"  Val:   {X_val_q.shape}")
print(f"  Test:  {X_test_q.shape}")
print(f"  Feature range: [{X_train_q.min():.3f}, {X_train_q.max():.3f}]")

## 2. Quantum Feature Maps

Feature maps encode classical data into quantum states. Different maps capture different data relationships.

In [ ]:
if QISKIT_AVAILABLE:
    # Compare different feature maps
    feature_map_types = [
        (FeatureMapType.ZZ, "ZZFeatureMap"),
        (FeatureMapType.Z, "ZFeatureMap"),
        (FeatureMapType.PAULI, "PauliFeatureMap"),
    ]
    
    print("Feature Map Comparison:")
    print("="*60)
    
    for fm_type, name in feature_map_types:
        info = get_feature_map_info(fm_type, N_QUBITS, reps=2)
        print(f"\n{name}:")
        for key, value in info.items():
            print(f"  {key}: {value}")

In [ ]:
if QISKIT_AVAILABLE:
    # Visualize ZZFeatureMap circuit
    fm = create_feature_map(
        n_qubits=N_QUBITS,
        feature_map_type=FeatureMapType.ZZ,
        reps=2,
        entanglement='linear'
    )
    
    print("ZZFeatureMap Circuit (4 qubits, 2 reps):")
    print(fm.draw(output='text', fold=100))

## 3. QSVM Training

Train QSVM with quantum kernel for binary classification.

In [ ]:
# Subsample for faster quantum computation
N_TRAIN = 200  # Quantum kernel computation is O(n^2)
N_TEST = 100

# Random subsample
np.random.seed(42)
train_idx = np.random.choice(len(X_train_q), N_TRAIN, replace=False)
test_idx = np.random.choice(len(X_test_q), N_TEST, replace=False)

X_train_sub = X_train_q[train_idx]
y_train_sub = data['y_train'][train_idx]
X_test_sub = X_test_q[test_idx]
y_test_sub = data['y_test'][test_idx]

print(f"Subsampled data for quantum:")
print(f"  Train: {X_train_sub.shape}")
print(f"  Test:  {X_test_sub.shape}")

In [ ]:
# Configure QSVM
config = QSVMConfig(
    n_qubits=N_QUBITS,
    feature_map_type=FeatureMapType.ZZ,
    feature_map_reps=2,
    entanglement='linear',
    use_classical_fallback=True,
    max_samples_quantum=N_TRAIN,
)

print("QSVM Configuration:")
print(f"  Qubits: {config.n_qubits}")
print(f"  Feature map: {config.feature_map_type.value}")
print(f"  Reps: {config.feature_map_reps}")
print(f"  Entanglement: {config.entanglement}")

In [ ]:
# Train QSVM
print("Training QSVM...")
qsvm = QSVM(config)
result = qsvm.fit_evaluate(X_train_sub, y_train_sub, X_test_sub, y_test_sub)

print("\n" + "="*60)
print("QSVM RESULTS")
print("="*60)
print(f"Used quantum kernel: {result.used_quantum}")
print(f"Accuracy:           {result.accuracy:.4f}")
print(f"F1 Score:           {result.f1_score:.4f}")
print(f"Training time:      {result.train_time:.2f}s")
print(f"Kernel time:        {result.kernel_computation_time:.2f}s")
print(f"Support vectors:    {result.n_support_vectors}")

## 4. Quantum vs Classical Comparison

In [ ]:
# Compare quantum and classical SVM
print("Running comparison...")
quantum_result, classical_result = compare_quantum_classical(
    X_train_sub, y_train_sub,
    X_test_sub, y_test_sub,
    n_qubits=N_QUBITS
)

# Create comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score', 'Train Time (s)', 'Support Vectors'],
    'QSVM (Quantum)': [
        quantum_result.accuracy if quantum_result else 'N/A',
        quantum_result.f1_score if quantum_result else 'N/A',
        quantum_result.train_time if quantum_result else 'N/A',
        quantum_result.n_support_vectors if quantum_result else 'N/A',
    ],
    'SVM (Classical)': [
        classical_result.accuracy,
        classical_result.f1_score,
        classical_result.train_time,
        classical_result.n_support_vectors,
    ]
})

print("\n" + "="*60)
print("QUANTUM VS CLASSICAL COMPARISON")
print("="*60)
comparison

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Performance comparison
metrics = ['Accuracy', 'F1 Score']
quantum_vals = [quantum_result.accuracy, quantum_result.f1_score] if quantum_result else [0, 0]
classical_vals = [classical_result.accuracy, classical_result.f1_score]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, quantum_vals, width, label='QSVM', color='#9b59b6')
axes[0].bar(x + width/2, classical_vals, width, label='Classical SVM', color='#3498db')
axes[0].set_ylabel('Score')
axes[0].set_title('Performance Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Training time comparison
times = [quantum_result.train_time if quantum_result else 0, classical_result.train_time]
labels = ['QSVM', 'Classical SVM']
colors = ['#9b59b6', '#3498db']

axes[1].bar(labels, times, color=colors)
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Training Time')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Kernel Matrix Visualization

In [ ]:
if QISKIT_AVAILABLE and qsvm._used_quantum:
    # Compute kernel matrix for a small subset
    n_vis = 30
    X_vis = X_train_sub[:n_vis]
    y_vis = y_train_sub[:n_vis]
    
    print(f"Computing quantum kernel matrix for {n_vis} samples...")
    K = qsvm.get_kernel_matrix(X_vis)
    
    # Sort by class for better visualization
    sort_idx = np.argsort(y_vis)
    K_sorted = K[sort_idx][:, sort_idx]
    y_sorted = y_vis[sort_idx]
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(K_sorted, cmap='RdBu_r', vmin=0, vmax=1)
    
    # Add class boundary line
    n_class0 = (y_sorted == 0).sum()
    ax.axhline(y=n_class0 - 0.5, color='black', linestyle='--', linewidth=2)
    ax.axvline(x=n_class0 - 0.5, color='black', linestyle='--', linewidth=2)
    
    plt.colorbar(im, label='Kernel Value (Fidelity)')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Sample Index')
    ax.set_title('Quantum Kernel Matrix\n(sorted by class: Normal | Attack)')
    
    # Add class labels
    ax.text(n_class0/2, -2, 'Normal', ha='center', fontsize=12, fontweight='bold')
    ax.text(n_class0 + (n_vis - n_class0)/2, -2, 'Attack', ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nKernel matrix statistics:")
    print(f"  Mean: {K.mean():.4f}")
    print(f"  Std:  {K.std():.4f}")
    print(f"  Min:  {K.min():.4f}")
    print(f"  Max:  {K.max():.4f}")
else:
    print("Quantum kernel not available for visualization")

## 6. Feature Map Comparison Experiment

In [ ]:
if QISKIT_AVAILABLE:
    # Compare different feature maps
    results_fm = []
    
    for fm_type in [FeatureMapType.ZZ, FeatureMapType.Z]:
        print(f"\nTesting {fm_type.value}...")
        
        config = QSVMConfig(
            n_qubits=N_QUBITS,
            feature_map_type=fm_type,
            feature_map_reps=2,
            use_classical_fallback=True,
        )
        
        qsvm = QSVM(config)
        result = qsvm.fit_evaluate(X_train_sub, y_train_sub, X_test_sub, y_test_sub)
        
        results_fm.append({
            'Feature Map': fm_type.value,
            'Accuracy': result.accuracy,
            'F1 Score': result.f1_score,
            'Train Time': result.train_time,
            'Used Quantum': result.used_quantum,
        })
    
    results_fm_df = pd.DataFrame(results_fm)
    print("\n" + "="*60)
    print("FEATURE MAP COMPARISON")
    print("="*60)
    results_fm_df

## Summary

### QSVM Performance

| Metric | QSVM | Classical SVM |
|--------|------|---------------|
| Accuracy | ~85-95% | ~90-95% |
| F1 Score | ~85-95% | ~90-95% |
| Training Time | Longer | Faster |

### Key Observations

1. **Quantum kernel computation** scales as O(n²) - limiting practical sample sizes
2. **ZZFeatureMap** typically performs well due to entanglement capturing feature correlations
3. **Current NISQ limitations**: 4 qubits requires aggressive feature reduction

### When Quantum May Win

- Complex, high-dimensional data where classical kernels fail
- True zero-day patterns not captured by classical models
- Future fault-tolerant quantum computers with more qubits

**Next**: QI-VAE for anomaly detection in `04_qivae_implementation.ipynb`